# S6 · Discover the Snowflake estate

Reads the source through the AIDP connector and writes `discovery_manifest.json`. Costs TWO `INFORMATION_SCHEMA` queries for the whole database, which is what makes a hundred-thousand-table estate feasible; a `DESCRIBE` per object does not scale and is not the path.

**Read-only against Snowflake.** The transport refuses any verb that is not SELECT/SHOW/DESCRIBE/DESC/WITH/EXPLAIN before it reaches the source.

---

*Generated from `engine/dataplane/00_discover_snowflake.py` by `engine/target/stage_notebooks.py`. Regenerate with `snowmig.py build-notebooks`; do not hand-edit — an edit here is overwritten on the next build. Change the source instead.*

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────
# Edit these, then run the notebook top to bottom. Every value is a
# plain Python literal: `None` means the flag is not passed at all,
# and `True` means a bare switch is passed.
#
# Scope and mode are INPUTS. To migrate less, change a value here --
# never edit the stage logic below to make it cover less.
PARAMS = {
    'source-mode': 'connector',
    'source-config': None,
    'source-catalog': None,
    'session-schema': None,
    'schemas': None,
    'exclude-schemas': None,
    'reports-dir': None,
    'backup-dir': None,
    'output-dir': None,
    'force': False,
}

# ── WORKFLOW PARAMETERS WIN ────────────────────────────────────
# A job task's `parameters` ({name, value}) override the literals
# above, by the same names (`schema`, `mode`, `dry-run`, ...). Read
# with oidlUtils.parameters.getParameter -- resolved by the AIDP
# runtime, never imported. The environment is only a fallback. This
# is how ONE notebook serves one workflow per schema: each task
# passes its own `schema`.
import os as _os
_MISS = '\x00__no_parameter__'
_SWITCHES = ['force']
_MULTI = ['exclude-schemas', 'schemas']
_CHOICES = {'source-mode': ('connector', 'external-catalog')}
_TRUE = ('true', 'yes', 'on', '1')
_FALSE = ('false', 'no', 'off', '0', '')


def _spellings(name):
    # `dry-run` as an operator may type it in the console: dry-run,
    # dry_run, dryRun, dryrun, and the upper-case forms. A spelling
    # nobody reads is a silent default -- for dry-run, a real write.
    parts = name.split('-')
    snake = '_'.join(parts)
    camel = parts[0] + ''.join(p[:1].upper() + p[1:] for p in parts[1:])
    flat = ''.join(parts)
    return list(dict.fromkeys((name, snake, camel, flat, name.upper(),
                               snake.upper(), flat.upper())))


def _workflow_param(name):
    for n in _spellings(name):
        try:
            v = oidlUtils.parameters.getParameter(n, _MISS)  # noqa: F821
        except NameError:  # outside AIDP: no oidlUtils
            v = _MISS
        if v != _MISS and v is not None and str(v).strip() != '':
            return str(v).strip()
    # The environment is a fallback only, by the plain spellings: a
    # generic name (SCHEMA, DRYRUN) could be any stray cluster variable.
    for n in dict.fromkeys((name, name.replace('-', '_'),
                            name.replace('-', '_').upper())):
        v = _os.environ.get(n)
        if v is not None and str(v).strip() != '':
            return str(v).strip()
    return None


PARAMS_FROM_WORKFLOW = {}
for _key in list(PARAMS):
    _v = _workflow_param(_key)
    if _v is None:
        continue
    if _key in _SWITCHES:
        if _v.lower() in _TRUE:
            _v = True
        elif _v.lower() in _FALSE:
            _v = False
        else:
            # A typo is not guessed either way: read as False, a
            # `dry-run` typo would turn a dry run into a real write.
            raise ValueError(
                f'workflow parameter {_key}={_v!r}: `{_key}` is a '
                f'switch; give true or false')
    elif _key in _MULTI:
        _v = [p.strip() for p in _v.split(',') if p.strip()]
    _bad = [c for c in (_v if isinstance(_v, list) else [_v])
            if _key in _CHOICES and c not in _CHOICES[_key]]
    if _bad:
        raise ValueError(
            f'workflow parameter {_key}={_bad[0]!r}: `{_key}` takes '
            f"one of {', '.join(_CHOICES[_key])}")
    PARAMS[_key] = PARAMS_FROM_WORKFLOW[_key] = _v
print('from the workflow:', PARAMS_FROM_WORKFLOW)


def _argv(params, repeated=()):
    """PARAMS -> argv. None is omitted; True is a bare switch; a list
    follows its flag, or repeats the flag per value for a name in
    `repeated` (an append-style flag)."""
    argv = []
    for key, value in params.items():
        if value is None or value is False:
            continue
        if value is True:
            argv.append(f'--{key}')
            continue
        values = value if isinstance(value, (list, tuple)) else [value]
        if key in repeated:
            for v in values:
                argv.extend((f'--{key}', str(v)))
        else:
            argv.append(f'--{key}')
            argv.extend(str(v) for v in values)
    return argv


ARGV = _argv(PARAMS)
print('arguments:', ARGV)

## Shared source helpers

Inlined from `engine/dataplane/snowmig_source.py` so this notebook runs with nothing else uploaded beside it.

In [ ]:
"""How the migration scripts READ Snowflake from inside AIDP. Two modes.

LIVE-VERIFIED 2026-09-16 on a real cluster (Spark 3.5, AIDP 4.x):

  connector (default)  spark.read.format("aidataplatform") with
                       type=SNOWFLAKE. Talked to the account, read a table
                       (87 rows x 9 cols) and ran a pushdown query
                       (`current_user()` = the service user). Needs NO extra
                       cluster library — the format is built in — and needs
                       NO successful catalog crawl.

  external-catalog     three-part names `catalog.schema.table` against a
                       registered EXTERNAL catalog. Cheaper (no per-read
                       Snowflake login) but it can only see what the CRAWLER
                       has already discovered, and on the validated
                       deployment the crawler failed with
                       "CONNECTOR_0067 ... Login has timed out" while the
                       connector above worked with the same credentials.

So: connector mode is the default because it is the one proven end to end,
and external-catalog mode is kept for a deployment whose crawl succeeds.

The option NAMES are the live-verified raw ones, which differ from the
Python helper's keyword names — `user.name` (not `user`), `database.name`,
`authentication.method` ∈ {Basic, KeyPair}, `private.key.content`. A wrong
name fails loud (`DATA_ACCESS_LAYER_0001 - Required spark option ... was not
provided`), which is how these were established.

Credentials come from a CONFIG FILE, never from arguments: the same rule the
control plane follows. Point --source-config at a JSON file shaped like
snowmig-config.example.yaml (JSON, on the workspace mount).
"""
from __future__ import annotations

import gzip
import hashlib
import json
import pathlib
import re

__all__ = ["SOURCE_MODES", "SourceConfigError", "SnowflakeSource",
           "write_step_output", "read_plan_json",
           "load_source_config"]

SOURCE_MODES = ("connector", "external-catalog")

AIDP_FORMAT = "aidataplatform"


# The verbs this transport may send. Deliberately narrower than the
# control-plane transport's list: WITH is absent, because following a CTE to
# the statement it prefixes needs the engine's lexer, which does not exist on
# a cluster. A read that needs a CTE can be written as a subquery.
PUSHDOWN_READ_VERBS = ("SELECT", "SHOW", "DESCRIBE", "DESC", "EXPLAIN")


class SourceWriteRefused(PermissionError):
    """A statement that is not a read was handed to the pushdown transport."""


def _code_only(sql: str) -> str:
    """`sql` with string literals and comments blanked, length preserved.

    A `;` inside a literal is data, not a statement boundary, and `--` or
    `//` inside one is not a comment. Blanking rather than deleting keeps offsets, so the
    scan cannot be confused about where anything starts.

    Lexed as SNOWFLAKE lexes: a backslash escapes only inside a '...'
    string; a "..." identifier ends at the first quote that is not doubled.
    Honouring a backslash there too made `"a\\"; delete from T` one
    identifier to this scan and two statements to Snowflake -- the guard
    failed open.
    """
    out = []
    i, n = 0, len(sql)
    while i < n:
        c = sql[i]
        two = sql[i:i + 2]
        if c in ("'", '"'):
            out.append(" ")
            i += 1
            closed = False
            while i < n:
                if c == "'" and sql[i] == "\\" and i + 1 < n:  # escape
                    out.append("  ")
                    i += 2
                    continue
                if sql[i] == c:
                    if sql[i:i + 2] == c * 2:           # doubled = literal
                        out.append("  ")
                        i += 2
                        continue
                    out.append(" ")
                    i += 1
                    closed = True
                    break
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            if not closed:
                raise SourceWriteRefused(
                    f"an unclosed {c} literal; refused (fails closed)")
            continue
        if two == "$$":
            out.append("  ")
            i += 2
            while i < n and sql[i:i + 2] != "$$":
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            if i >= n:
                raise SourceWriteRefused(
                    "an unclosed $$ string; refused (fails closed)")
            out.append("  ")
            i += 2
            continue
        # `//` is a line comment in Snowflake exactly as `--` is. Missing it
        # let an apostrophe in `// it's` open a phantom literal that hid the
        # `;` on the next line -- two statements read as one.
        if two in ("--", "//"):
            while i < n and sql[i] != "\n":
                out.append(" ")
                i += 1
            continue
        if two == "/*":
            # Snowflake ends a block comment at the FIRST `*/` (no nesting),
            # as the engine's lexer does. Counting depth made
            # `select 1 /* /* */ ; delete from t` one statement here and
            # two to Snowflake. An unclosed comment is refused.
            i += 2
            out.append("  ")
            while i < n and sql[i:i + 2] != "*/":
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            if i >= n:
                raise SourceWriteRefused(
                    "an unclosed /* comment; refused (fails closed)")
            out.append("  ")
            i += 2
            continue
        out.append(c)
        i += 1
    return "".join(out)


def assert_pushdown_read_only(sql: str) -> None:
    """Refuse anything that is not a single read. Fails closed.

    The cluster-side counterpart of the control plane's `assert_read_only`:
    the credential the notebook holds may well be able to write, and the
    only thing standing between a migration and a modified SOURCE is this
    check.
    """
    code = _code_only(sql or "")
    if "->>" in code:
        # Snowflake's flow operator chains another statement into the same
        # request: `select 1 ->> delete from t` would pass a leading-verb
        # check. Nothing this plugin generates uses it.
        raise SourceWriteRefused(
            "the ->> flow operator chains statements; refused")
    statements = [s for s in code.split(";") if s.strip()]
    if not statements:
        raise SourceWriteRefused(
            f"empty statement refused; this transport is read-only against "
            f"Snowflake (allowed: {', '.join(PUSHDOWN_READ_VERBS)})")
    if len(statements) > 1:
        raise SourceWriteRefused(
            f"{len(statements)} statements in one pushdown; refused. A "
            f"second statement is how a write rides along behind a read.")
    verb = statements[0].split()[0].upper() if statements[0].split() else ""
    if verb == "WITH":
        raise SourceWriteRefused(
            "a CTE is refused by this transport: deciding whether `WITH ... "
            "INSERT` is a read needs the engine's scanner, which does not "
            "run on the cluster. Write the CTE as a subquery.")
    if verb not in PUSHDOWN_READ_VERBS:
        raise SourceWriteRefused(
            f"{verb or 'unrecognised statement'} refused: this transport is "
            f"read-only against Snowflake, whatever the credential allows "
            f"(allowed: {', '.join(PUSHDOWN_READ_VERBS)}).")


class SourceConfigError(ValueError):
    """The source config is missing, unreadable or incomplete."""


def q(identifier: str) -> str:
    """Backtick-quote one Spark identifier."""
    return "`" + str(identifier).replace("`", "``") + "`"


def _sql_ident(identifier: str) -> str:
    """Double-quote one SNOWFLAKE identifier (the pushdown runs there)."""
    return '"' + str(identifier).replace('"', '""') + '"'


# An identifier Snowflake reads WITHOUT quotes, and so resolves upper-cased.
_UNQUOTED_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")


def _database_name(name: str) -> str:
    """The name Snowflake resolves the config's database to, unquoted.

    `database: snowmig_db` is an unquoted identifier to the connector, so it
    names SNOWMIG_DB. `database: '"MyDb"'` -- written with its double quotes,
    as Snowflake SQL would -- names the mixed-case MyDb exactly. Any other
    value that is not a plain identifier can only have been meant verbatim.
    Every statement that names the database goes through here, so no two
    of them can address different databases.
    """
    text = str(name).strip()
    if len(text) >= 2 and text[0] == text[-1] == '"':
        return text[1:-1].replace('""', '"')
    if _UNQUOTED_IDENT.match(text):
        return text.upper()
    return text


def _sql_database(name: str) -> str:
    """The config's database, quoted as the name Snowflake resolves it to."""
    return _sql_ident(_database_name(name))


# The copy expressions for a column whose live type is not the planned one,
# used only under `mapping.source_type_drift: convert`. A mirror of
# snowflake_source/dialect/types.copy_expressions -- this module is inlined
# into every notebook and cannot import the engine -- held to it by a parity
# test. The TIME precision is the live read's default (all nine digits);
# GEOGRAPHY reads as GeoJSON, as a plan without a recorded mode does.
_DRIFT_NUMERIC = {"NUMBER", "DECIMAL", "NUMERIC", "INT", "INTEGER", "BIGINT",
                  "SMALLINT", "TINYINT", "BYTEINT"}
_DRIFT_FLOATS = {"FLOAT", "FLOAT4", "FLOAT8", "DOUBLE", "REAL",
                 "DOUBLE PRECISION"}
_DRIFT_ZONED = ("TIMESTAMP_TZ", "TIMESTAMP_LTZ", "TIMESTAMP")
_DRIFT_JSON = ("VARIANT", "OBJECT", "ARRAY", "MAP")
_DRIFT_GEO = ("GEOGRAPHY", "GEOMETRY")


def live_copy_expressions(data_type: str, target_type: str, *,
                          name: str) -> tuple[str, str]:
    """(read_expr, convert_expr) for `name` read as its LIVE `data_type`
    and converted to the EXISTING target column's `target_type`."""
    col = '"' + str(name).replace('"', '""') + '"'
    out = "`" + str(name).replace("`", "``") + "`"
    key = str(data_type or "").strip().upper().split("(")[0].strip()
    target = str(target_type or "")
    upper = target.upper()
    if upper.startswith(("ARRAY<", "MAP<", "STRUCT<")):
        read = f"{col}::ARRAY::VARCHAR" if key == "VECTOR" \
            else f"{col}::VARIANT::VARCHAR"
        schema = target if "STRUCT<" in upper else target.lower()
        literal = "'" + schema.replace("\\", "\\\\").replace("'", "\\'") + "'"
        return read, f"from_json({out}, {literal})"
    if key in _DRIFT_NUMERIC:
        return f"{col}::VARCHAR", f"CAST({out} AS {target})"
    if key in _DRIFT_FLOATS:
        return f"TO_VARCHAR({col}, 'TME')", f"CAST({out} AS {target})"
    if key == "TIME":
        return f"TO_VARCHAR({col}, 'HH24:MI:SS.FF9')", out
    if key == "TIMESTAMP_NTZ":
        return (f"TO_VARCHAR({col}, 'YYYY-MM-DD HH24:MI:SS.FF9')",
                f"CAST({out} AS {target})")
    if key in _DRIFT_ZONED:
        return (f"TO_VARCHAR({col}, 'YYYY-MM-DD\"T\"HH24:MI:SS.FF9TZH:TZM')",
                f"CAST({out} AS {target})")
    if key in _DRIFT_GEO:
        return f"ST_ASGEOJSON({col})::VARCHAR", out
    if key in _DRIFT_JSON and upper == "STRING":
        return f"TO_JSON({col}::VARIANT)", out
    return col, out


def _sql_literal(value: str) -> str:
    """Escape a string literal for Snowflake SQL.

    The backslash first: Snowflake reads it as an escape inside '...', so a
    table named `a\\` made `'a\\'` an unterminated literal that swallowed the
    SQL after it.
    """
    return str(value).replace("\\", "\\\\").replace("'", "''")


# The /Workspace mount can serve a report that was just written before its
# bytes are all there: an incomplete read is retried briefly, then fails
# loudly, naming the file. Shared by every stage that re-reads a report.
REPORT_READ_TRIES = 5
REPORT_READ_WAIT = 2.0


def read_report_json(path: pathlib.Path) -> dict:
    """A report's JSON, retried briefly when the read comes back incomplete;
    still invalid after that is a loud failure that names the file."""
    import time
    last = None
    for attempt in range(REPORT_READ_TRIES):
        try:
            return json.loads(pathlib.Path(path).read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            last = exc
            if attempt + 1 < REPORT_READ_TRIES:
                time.sleep(REPORT_READ_WAIT)
    raise ValueError(f"{pathlib.Path(path).name} is not valid JSON after "
                     f"{REPORT_READ_TRIES} reads ({last}); it was not "
                     f"overwritten -- inspect it before re-running")


def load_source_config(path: str | pathlib.Path) -> dict:
    """Read the Snowflake connection config (JSON) from the workspace."""
    p = pathlib.Path(path).expanduser()
    try:
        text = p.read_text(encoding="utf-8")
    except OSError as exc:
        raise SourceConfigError(
            f"source config not readable at {p}: {exc.strerror}") from exc
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as exc:
            raise SourceConfigError(
                f"{p} is YAML but PyYAML is not on the cluster; write the "
                f"config as JSON instead") from exc
        try:
            data = yaml.safe_load(text) or {}
        except yaml.YAMLError as exc:
            # The parser's message quotes the offending line, which can be
            # the password line: only its position is reported.
            mark = getattr(exc, "problem_mark", None)
            where = f" at line {mark.line + 1}" if mark is not None else ""
            raise SourceConfigError(
                f"{p} is not valid YAML{where}; the line is withheld, "
                f"since it may hold the credential") from None
    else:
        try:
            data = json.loads(text) if text.strip() else {}
        except json.JSONDecodeError as exc:
            raise SourceConfigError(
                f"{p} is not valid JSON at line {exc.lineno}; the line is "
                f"withheld, since it may hold the credential") from None
    if not isinstance(data, dict):
        raise SourceConfigError(f"{p}: expected a mapping at the top level")
    # THE MIGRATION CONFIG IS ONE FILE FOR BOTH ENDS: the Snowflake connection
    # nested under `snowflake:`, the AIDP coordinates under `aidp:`. That is
    # the file `provision --source-config` reads, and it uploads the
    # `snowflake:` block as JSON (`plan/<stem>.json`) -- so the shape that
    # reaches the mount is the nested one, and reading the top level for
    # `account` found nothing but the envelope key. The failure surfaced on
    # the cluster, as every required field missing at once, which reads
    # like a broken credential rather than a config one level too deep.
    nested = data.get("snowflake")
    if isinstance(nested, dict):
        return dict(nested)
    return data


class SnowflakeSource:
    """Read-only access to the Snowflake source, in either mode.

    Nothing here can write: every method issues a read, and the connector is
    read-only in AIDP 4.0 by Oracle's own statement.
    """

    def __init__(self, spark, *, mode: str = "connector",
                 config: dict | None = None,
                 external_catalog: str | None = None,
                 session_schema: str | None = None):
        if mode not in SOURCE_MODES:
            raise SourceConfigError(
                f"unknown source mode {mode!r}; expected one of "
                f"{list(SOURCE_MODES)}")
        self.spark = spark
        self.mode = mode
        self.external_catalog = external_catalog
        self._options: dict[str, str] = {}
        # A REAL schema, used only to scope a pushdown session. The connector
        # validates this option against the schemas it can see and rejects
        # INFORMATION_SCHEMA itself with DATA_ACCESS_LAYER_0031 -- but a
        # pushdown query scoped to a real schema may reference
        # INFORMATION_SCHEMA freely (both established live).
        self.session_schema = session_schema or (config or {}).get("schema")

        if mode == "external-catalog":
            if not external_catalog:
                raise SourceConfigError(
                    "external-catalog mode needs --source-catalog")
            return

        cfg = config or {}
        missing = [k for k in ("account", "warehouse", "database", "user",
                               "auth") if not cfg.get(k)]
        if missing:
            raise SourceConfigError(
                "source config is missing required field(s): "
                + ", ".join(sorted(missing)))
        account = str(cfg["account"]).strip()
        # The connector wants the account/server URL host, which is what the
        # Snowflake console calls "Account/Server URL".
        host = str(cfg.get("host")
                   or f"{account}.snowflakecomputing.com").strip()
        opts = {
            "type": "SNOWFLAKE",
            "host": host,
            "port": str(cfg.get("port") or 443),
            "database.name": str(cfg["database"]).strip(),
            "user.name": str(cfg["user"]).strip(),
            "warehouse": str(cfg["warehouse"]).strip(),
        }
        if cfg.get("role"):
            opts["role"] = str(cfg["role"]).strip()

        # A secret may be inline in the one config file, or in a file the
        # config points at. On a cluster the inline form is usually the only
        # one available, since the workspace mount carries the config but not
        # the operator's home directory.
        def secret(inline, path_field):
            if cfg.get(inline):
                return str(cfg[inline])
            if cfg.get(path_field):
                return pathlib.Path(
                    str(cfg[path_field])).expanduser().read_text(encoding="utf-8").strip()
            return None

        auth = str(cfg["auth"]).strip().lower()
        if auth == "keypair":
            key = secret("private_key", "key_path")
            if not key:
                raise SourceConfigError(
                    "auth: keypair needs `private_key` (inline) or `key_path`")
            opts["authentication.method"] = "KeyPair"
            opts["private.key.content"] = key
            passphrase = secret("key_passphrase", "key_passphrase_path")
            if passphrase:
                opts["private.key.pass.phrase"] = passphrase
        elif auth == "password":
            password = secret("password", "password_path")
            if not password:
                raise SourceConfigError(
                    "auth: password needs `password` (inline) or "
                    "`password_path`")
            opts["authentication.method"] = "Basic"
            opts["password"] = password
        else:
            raise SourceConfigError(
                f"auth {auth!r} is not supported by the AIDP Snowflake "
                f"connector; use keypair (preferred) or password")
        self._options = opts

    # -- describing the estate ------------------------------------------

    def database(self) -> str | None:
        return self._options.get("database.name")

    def qualified(self, schema: str, table: str) -> str:
        """`"DB"."SCHEMA"."TABLE"` for a pushdown, the database from the config.

        LIVE 2026-09-29: the pushdown session has NO current schema, whatever
        the connector's `schema` option says, so an unqualified `"T"` fails
        with "Object does not exist" -- the root cause of the batched count's
        CONNECTOR_0099. Every table a pushdown names goes through here.
        Schema and table keep their exact case: they come from
        INFORMATION_SCHEMA, where the case is the name.
        """
        database = self.database()
        if not database:
            raise SourceConfigError(
                "a qualified pushdown needs the source database; add "
                "`database:` to the source config")
        return (f"{_sql_database(database)}.{_sql_ident(schema)}."
                f"{_sql_ident(table)}")

    def pushdown(self, sql: str, *, schema: str | None = None):
        """Run `sql` IN SNOWFLAKE and return a DataFrame.

        Refuses anything that is not a single read statement, whatever the
        credential allows -- see `assert_pushdown_read_only`.

        Connector mode only. The `schema` option must name a REAL schema:
        the connector rejects INFORMATION_SCHEMA there with
        DATA_ACCESS_LAYER_0031, while a query scoped to a real schema may
        reference INFORMATION_SCHEMA freely. Both established live.
        """
        # Before the mode check and before anything reaches Spark: the
        # refusal must not depend on configuration being right.
        assert_pushdown_read_only(sql)
        if self.mode != "connector":
            raise SourceConfigError(
                "pushdown is connector-mode only; external-catalog mode has "
                "no Snowflake session to push into")
        scope = schema or self.session_schema
        if not scope:
            raise SourceConfigError(
                "connector pushdown needs a REAL schema to scope the "
                "session: add `schema:` to the source config or pass "
                "--session-schema. INFORMATION_SCHEMA is not accepted there.")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", scope)
                .option("pushdown.sql", sql)
                .load())

    def source_counts(self, schema: str, tables: list[str], *,
                      chunk: int = 50) -> dict[str, int]:
        """`{table: COUNT(*)}` for many tables in ONE round trip per chunk.

        Connector mode opens a Snowflake session per read, and the copy needs
        a source count twice per table (before, and again after, to catch a
        source that moved during the copy). Per-table counting therefore cost
        more session setup than the copy itself on a live run. A single
        UNION ALL answers a whole schema instead; it is chunked because a
        statement with thousands of branches is its own problem.

        Every branch names its table "DB"."SCHEMA"."TABLE" (see `qualified`):
        unqualified, the live pushdown session refused every chunk with
        CONNECTOR_0099 and the copy fell back to a count per table.
        Qualified, 50 tables answered in one query in 25 s (2026-09-29).

        External-catalog mode has no session to amortise, so it falls back to
        a count per table -- correct either way, and the caller does not care.
        """
        out: dict[str, int] = {}
        if self.mode != "connector":
            for table in tables:
                out[table] = self.read_table(schema, table).count()
            return out
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            sql = " union all ".join(
                # The literal is the table's own name, so one query can carry
                # many counts and still say which is which. Both the literal
                # and the identifier are escaped: one apostrophe in a table
                # name would otherwise break the whole chunk.
                f"select '{_sql_literal(t)}' as SNOWMIG_TABLE, "
                f"count(*) as SNOWMIG_N from {self.qualified(schema, t)}"
                for t in batch)
            for row in self.pushdown(sql, schema=schema).collect():
                data = row.asDict()
                out[str(data["SNOWMIG_TABLE"])] = int(data["SNOWMIG_N"])
        return out

    def live_columns(self, schema: str, tables: list[str], *,
                     chunk: int = 200) -> dict[str, dict[str, str]]:
        """`{table: {column: type}}` for many tables, from INFORMATION_SCHEMA.

        The copy compares the LIVE source's columns with the target's before
        it moves a row. The connector's table read answered that from its
        own metadata lookup, at 126-241 s per table (2026-09-29); one
        INFORMATION_SCHEMA.COLUMNS query answers a chunk of tables instead.
        Types are what the copy's pre-flight compares: NUMBER(p,s) becomes
        `decimal(p,s)` (the DECIMAL check), everything else is Snowflake's
        own type name lower-cased. A table the query does not list is ABSENT
        from the result, never an empty list -- the caller says so.
        Connector mode only.
        """
        out: dict[str, dict[str, str]] = {}
        database = self.database()
        if not database:
            raise SourceConfigError(
                "reading INFORMATION_SCHEMA needs the source database; add "
                "`database:` to the source config")
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            names = ", ".join(f"'{_sql_literal(t)}'" for t in batch)
            sql = ("select TABLE_NAME, COLUMN_NAME, DATA_TYPE, "
                   "NUMERIC_PRECISION, NUMERIC_SCALE, ORDINAL_POSITION "
                   f"from {_sql_database(database)}.INFORMATION_SCHEMA.COLUMNS "
                   f"where TABLE_SCHEMA = '{_sql_literal(schema)}' "
                   f"and TABLE_NAME in ({names}) "
                   "order by TABLE_NAME, ORDINAL_POSITION")
            rows = sorted((r.asDict() for r in
                           self.pushdown(sql, schema=schema).collect()),
                          key=lambda d: (str(d["TABLE_NAME"]),
                                         int(d["ORDINAL_POSITION"] or 0)))
            for data in rows:
                kind = str(data.get("DATA_TYPE") or "").strip()
                if kind.upper() in ("NUMBER", "DECIMAL", "NUMERIC") and \
                        data.get("NUMERIC_PRECISION") is not None:
                    kind = (f"decimal({int(data['NUMERIC_PRECISION'])},"
                            f"{int(data.get('NUMERIC_SCALE') or 0)})")
                out.setdefault(str(data["TABLE_NAME"]), {})[
                    str(data["COLUMN_NAME"])] = kind.lower()
        return out

    def read_columns(self, schema: str, table: str,
                     select: list[tuple[str, str]]):
        """ONE qualified pushdown: `SELECT <expr> AS "<name>", ... FROM
        "DB"."SCHEMA"."TABLE"`.

        `select` is `[(name, read_expr)]`: a Snowflake expression over the
        quoted source column, unaliased. Each is aliased to the column's
        exact name, so the Spark side sees the source's names. Live
        2026-09-29 this cost ~8.5 s a table where the connector's table read
        cost 126-241 s, and the exact reads it carries (`::VARCHAR`,
        `TO_VARCHAR(.., FF9)`, `::ARRAY::VARCHAR`) are what keep NUMBER,
        TIME/TIMESTAMP fractions and VECTOR/MAP/OBJECT columns intact.
        The whole statement goes through the read-only guard, as every
        pushdown does.
        """
        if not select:
            raise SourceConfigError(f"no columns to read from {schema}.{table}")
        items = ", ".join(f"{expr} as {_sql_ident(name)}"
                          for name, expr in select)
        return self.pushdown(
            f"select {items} from {self.qualified(schema, table)}",
            schema=schema)

    def source_sums(self, schema: str, table: str,
                    columns: list[str]) -> dict[str, str | None]:
        """`{column: exact total as text}` summed IN SNOWFLAKE, one pushdown.

        `select sum("C")::VARCHAR as "C", ... from "DB"."SCHEMA"."TABLE"`:
        Snowflake adds its own NUMBERs exactly, and `::VARCHAR` carries the
        total past the connector's typing, which cut NUMBER to ten
        significant digits live (2026-09-29). None is SQL NULL -- no rows,
        or only NULLs.
        """
        if not columns:
            return {}
        items = ", ".join(f"sum({_sql_ident(c)})::VARCHAR as {_sql_ident(c)}"
                          for c in columns)
        row = self.pushdown(
            f"select {items} from {self.qualified(schema, table)}",
            schema=schema).collect()[0].asDict()
        return {c: (None if row.get(c) is None else str(row[c]))
                for c in columns}

    def register_columns_view(self, schema: str, table: str, view: str,
                              select: list[tuple[str, str]]) -> str:
        """`read_columns` registered as a session temp view; its name."""
        self.read_columns(schema, table, select).createOrReplaceTempView(view)
        return q(view)

    def read_table(self, schema: str, table: str):
        """A DataFrame over one source table."""
        if self.mode == "external-catalog":
            return self.spark.table(
                f"{q(self.external_catalog)}.{q(schema)}.{q(table)}")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", schema)
                .option("table", table)
                .load())

    def register_temp_view(self, schema: str, table: str, view: str) -> str:
        """Expose a source table to SQL as a temp view, and return its name.

        INSERT ... SELECT needs the source addressable in SQL. In
        external-catalog mode the three-part name already is; in connector
        mode the DataFrame is registered as a session-local temp view, which
        is dropped by the caller.
        """
        if self.mode == "external-catalog":
            return f"{q(self.external_catalog)}.{q(schema)}.{q(table)}"
        self.read_table(schema, table).createOrReplaceTempView(view)
        return q(view)

    def drop_temp_view(self, view: str) -> None:
        if self.mode == "connector":
            self.spark.catalog.dropTempView(view)

    def describe(self) -> dict:
        """What this source is, for the report header. Never the credential."""
        out = {"mode": self.mode}
        if self.mode == "external-catalog":
            out["external_catalog"] = self.external_catalog
        else:
            out.update(session_schema=self.session_schema,
                       host=self._options.get("host"),
                       database=self._options.get("database.name"),
                       user=self._options.get("user.name"),
                       warehouse=self._options.get("warehouse"),
                       role=self._options.get("role"),
                       auth=self._options.get("authentication.method"))
        return out


# What `provision` puts under a plan file's plain name when the file itself
# was too large to upload and went up gzipped beside it (see
# target/provisioning.py COMPRESS_OVER_BYTES).
PLAN_POINTER_KEY = "snowmig_compressed_to"


def read_plan_json(path) -> dict:
    """A plan file `provision` pushed: plain JSON, or a pointer to its gzip
    copy in the same folder, which is read and checked against the digest.

    Raises ValueError naming the file when what is there is not a plan.
    Live 2026-09-29, a failed overwrite upload left ddl_plan.json EMPTY on
    the workspace, and the stage died on a bare JSONDecodeError at char 0.
    """
    path = pathlib.Path(path)
    raw = path.read_bytes()
    try:
        doc = json.loads(raw.decode("utf-8"))
    except ValueError as exc:
        raise ValueError(
            f"{path} is not valid JSON ({len(raw):,} bytes: {exc}). A failed "
            f"upload leaves the file empty or partial; re-run `provision "
            f"--execute` and check that its upload of this file succeeded"
        ) from None
    if not (isinstance(doc, dict) and doc.get(PLAN_POINTER_KEY)):
        return doc
    name = str(doc[PLAN_POINTER_KEY])
    if pathlib.PurePosixPath(name).name != name or "\\" in name:
        raise ValueError(f"{path}: the pointer names {name!r}, which is not "
                         f"a file beside it")
    target = path.with_name(name)
    try:
        data = gzip.decompress(target.read_bytes())
    except (OSError, EOFError) as exc:
        raise ValueError(f"{target} (the plan {path.name} points at) could "
                         f"not be read: {exc}; re-run `provision --execute`"
                         ) from None
    if doc.get("sha256") and hashlib.sha256(data).hexdigest() != doc["sha256"]:
        raise ValueError(f"{target} does not match the digest its pointer "
                         f"{path.name} records: the two were not pushed "
                         f"together; re-run `provision --execute`")
    return json.loads(data.decode("utf-8"))


def write_step_output(output_dir: str | None, name: str,
                      payload: dict) -> str | None:
    """Save one runbook step's values under the workspace's report/output.

    `name` carries the step (S06_discover.json, S10_structure.json, ...).
    An empty output_dir switches it off. Never raises: the step's own
    reports are already written, and a copy that cannot be made must not
    turn a successful stage into a failed one.
    """
    if not output_dir:
        return None
    import datetime as _dt
    import json as _json
    try:
        folder = pathlib.Path(output_dir)
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / name
        path.write_text(_json.dumps(
            {**payload, "written_at": _dt.datetime.now(
                _dt.timezone.utc).isoformat()}, indent=2, default=str))
        print(f"[output] {name} -> {path}", flush=True)
        return str(path)
    except OSError as exc:
        print(f"[output] {name} NOT written to {output_dir}: {exc}", flush=True)
        return None


## Stage logic

In [ ]:
#!/usr/bin/env python3
"""Discover the Snowflake estate from inside AIDP. Writes discovery_manifest.json.

USE `connector`. Discovering schemas and tables by walking three-part names
against an EXTERNAL catalog is the slow path and should not be the starting
point -- see the table below before reaching for it.

  connector (default)   ONE pushdown query pair against INFORMATION_SCHEMA
                        returns every schema, table and column in the
                        database. This is the mode that scales: a
                        200k-table estate costs a handful of queries, not a
                        DESCRIBE per object. It needs only the credentials
                        smoke already proved -- no catalog crawl. A table
                        holding a VECTOR / MAP / OBJECT / ARRAY column
                        costs one more read: its full column types, which
                        INFORMATION_SCHEMA does not carry, come from
                        GET_DDL, batched 50 tables to a qualified pushdown.

  external-catalog      SHOW SCHEMAS/TABLES + DESCRIBE per object against a
                        registered EXTERNAL catalog. Its cost grows with the
                        object count, so it does not finish at estate scale,
                        and it sees only what the crawler already discovered
                        -- one more precondition that has to be true first.
                        Use it only when a user explicitly asks.

The estate is READ-ONLY under both modes: the AIDP Snowflake connector is
read-only in 4.0, and an external catalog refuses DDL by contract. Nothing is
written anywhere except --reports-dir.

External-catalog mode is resumable per schema (each is flushed immediately);
connector mode returns the estate whole, so there is nothing to resume.

`--schemas` is pushed into the INFORMATION_SCHEMA queries as a predicate, not
applied after the fetch, and a scoped run MERGES into the manifest: the named
schemas are refreshed and every other schema is kept. A named schema that
returns no rows keeps its previous discovery, gains an error, and fails the
run; `--force` drops it instead. `--force` alone (no `--schemas`)
re-discovers the whole estate from an empty manifest.
"""
from __future__ import annotations

import argparse
import datetime
import json
import pathlib
import re
import sys
import traceback


# /Workspace is the live-verified mount of the workspace tree on cluster
# filesystems (probed 2026-09-16 on a real cluster).
DEFAULT_REPORTS_DIR = "/Workspace/backup-snowflake-migration/reports"
# The step's own values also go here, next to the accumulated run report
# the CLI publishes after every stage. `--output-dir ''` switches it off.
DEFAULT_OUTPUT_DIR = "/Workspace/report/output"
MANIFEST_NAME = "discovery_manifest.json"
# Runbook S6: the manifest is backed up, dated, BEFORE any later stage reads
# it. The reports/ copy is the working one and a re-run merges into it; the
# dated copy is what a later reader can trust was the input. It goes to the
# `backup/` folder beside `reports/` unless --backup-dir says otherwise.
BACKUP_FOLDER_NAME = "backup"

# Snowflake's own system schema: never a migration target.
_SYSTEM_SCHEMAS = {"information_schema"}

# `{where}` is the --schemas predicate, or nothing; `{flags}` is
# _KIND_FLAGS_SELECT, or nothing on the fallback read.
#
# TABLE_TYPE, IS_TRANSIENT and COMMENT are written to the manifest, not only
# read: the bridge maps them onto the same `source_metadata` keys a live
# `assess` takes from SHOW TABLES. TABLE_TYPE once chose only the tables or
# the views bucket, so an EVENT or EXTERNAL TABLE planned from a manifest
# was can_migrate while the laptop path refused it.
_TABLES_SQL = (
    "select TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE, IS_TRANSIENT, ROW_COUNT, "
    "BYTES, COMMENT{flags} "
    "from INFORMATION_SCHEMA.TABLES{where} order by TABLE_SCHEMA, TABLE_NAME")

# Dynamic, Iceberg and hybrid tables are BASE TABLEs by TABLE_TYPE; only
# these columns tell them apart. They are younger than the rest of the view,
# so an account that lacks one gets the narrower read and a recorded gap --
# not a whole discovery lost to `invalid identifier`, and not a guess.
_KIND_FLAGS = ("IS_DYNAMIC", "IS_ICEBERG", "IS_HYBRID")
_KIND_FLAGS_SELECT = "".join(f", {f}" for f in _KIND_FLAGS)

# COLUMN_DEFAULT, IDENTITY_* and COMMENT are here because the planning
# stages warn on them (R22, R23) and carry the comment into the CREATE
# TABLE. Without them a manifest-planned estate gets a DDL plan that is
# silent about defaults and identity columns that stop working at cutover,
# while the same estate planned from a laptop warns about both.
_COLUMNS_SQL = (
    "select TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME, ORDINAL_POSITION, "
    "DATA_TYPE, IS_NULLABLE, NUMERIC_PRECISION, NUMERIC_SCALE, "
    "CHARACTER_MAXIMUM_LENGTH, COLUMN_DEFAULT, IDENTITY_START, "
    "IDENTITY_INCREMENT, COMMENT, COLLATION_NAME, DATETIME_PRECISION "
    "from INFORMATION_SCHEMA.COLUMNS{where} "
    "order by TABLE_SCHEMA, TABLE_NAME, ORDINAL_POSITION")


def _schema_predicate(wanted: list[str] | None) -> str:
    """` where TABLE_SCHEMA in (...)` for --schemas, or an empty string.

    Pushed into the query rather than applied after the fetch. An unfiltered
    read over a large database's INFORMATION_SCHEMA can exceed Snowflake's
    result cap ("Information schema query returned too much data"), and a
    --schemas scope that only filtered client-side could do nothing about
    it. TABLE_SCHEMA is compared as a string value, so the names are
    literals, not identifiers.
    """
    if not wanted:
        return ""
    names = ", ".join(f"'{_sql_literal(s)}'" for s in wanted)
    return f" where TABLE_SCHEMA in ({names})"


def log(msg: str) -> None:
    print(f"[discover] {msg}", flush=True)


def fail(msg: str) -> int:
    """Report a refusal on BOTH streams and return 1.

    A notebook task captures stdout only: live, a script that exited 1 via a
    stderr-only message produced a job failure with NO explanation anywhere.
    """
    print(f"ERROR: {msg}", flush=True)
    print(f"error: {msg}", file=sys.stderr)
    return 1


def _plain(value):
    """JSON-safe scalar.

    Snowflake numerics arrive as `decimal.Decimal` through the connector and
    `json.dumps` refuses them ("Object of type Decimal is not JSON
    serializable") -- which happened AFTER a successful 1065-relation read,
    so the whole discovery was lost at the write. Integral values keep their
    exactness as ints; everything else becomes a string rather than a lossy
    float.
    """
    if value is None or isinstance(value, (int, str, bool)):
        return value
    try:
        as_int = int(value)
    except (TypeError, ValueError):
        return str(value)
    return as_int if as_int == value else str(value)


def _rows(df) -> list[dict]:
    return [r.asDict() for r in df.collect()]


def _first_key(row: dict, *candidates: str):
    """SHOW column names vary by engine version; take the first match rather
    than assuming one, and fail loudly when none is there."""
    for key in candidates:
        if key in row:
            return row[key]
    lowered = {k.lower(): v for k, v in row.items()}
    for key in candidates:
        if key.lower() in lowered:
            return lowered[key.lower()]
    raise KeyError(f"none of {candidates} in row: {sorted(row)}")


def _snowflake_type(col: dict) -> str:
    """The SOURCE type as Snowflake reports it, precision preserved.

    Deliberately NOT translated: translation belongs to the engine's type
    mapper, which refuses rather than guesses. This manifest records what IS.
    """
    base = str(col.get("DATA_TYPE") or "").upper()
    precision = _plain(col.get("NUMERIC_PRECISION"))
    scale = _plain(col.get("NUMERIC_SCALE"))
    length = _plain(col.get("CHARACTER_MAXIMUM_LENGTH"))
    if base in ("NUMBER", "DECIMAL", "NUMERIC") and precision is not None:
        return f"{base}({int(precision)},{int(scale or 0)})"
    if base in ("TEXT", "VARCHAR", "CHAR", "STRING") and length is not None:
        return f"{base}({int(length)})"
    return base


# Columns whose INFORMATION_SCHEMA type hides what the engine's mapper needs:
# `VECTOR`, `MAP`, `OBJECT`, `ARRAY` carry no element types there (live
# 2026-09-29), and a TIME/TIMESTAMP with no DATETIME_PRECISION carries no
# fraction. The SAME rule as `snowflake_source.dialect.types.
# needs_type_detail`, which this script cannot import (it is uploaded as a
# standalone file); tests/test_type_detail.py holds the two together.
_TYPE_DETAIL_BASES = ("VECTOR", "MAP", "OBJECT", "ARRAY", "GEOGRAPHY",
                      "GEOMETRY")
_TIME_BASES = ("TIME", "TIMESTAMP", "TIMESTAMP_NTZ", "TIMESTAMP_LTZ",
               "TIMESTAMP_TZ")
# GET_DDL calls per pushdown. The live 50-table qualified UNION ALL count
# answered in 25 s; a pushdown per table costs ~8.5 s each.
_DDL_CHUNK = 50


def _needs_type_detail(data_type, datetime_precision=None) -> bool:
    base = str(data_type or "").strip().upper().split("(")[0].strip()
    if base in _TYPE_DETAIL_BASES:
        return True
    return base in _TIME_BASES and datetime_precision in (None, "")


def _scan_to(text: str, start: int, stop: str) -> int:
    """Index of the first `stop` character at depth 0 from `start`, outside
    string literals and quoted identifiers; len(text) when there is none.

    Snowflake's lexing: a backslash escapes inside '...', a doubled quote is
    a literal quote in either kind.
    """
    depth, i, n = 0, start, len(text)
    while i < n:
        c = text[i]
        if c in ("'", '"'):
            i += 1
            while i < n:
                if c == "'" and text[i] == "\\":
                    i += 2
                    continue
                if text[i] == c:
                    if text[i:i + 2] == c * 2:
                        i += 2
                        continue
                    break
                i += 1
        elif c == "(" and "(" in stop and depth == 0:
            return i
        elif c == "(":
            depth += 1
        elif c == ")":
            if depth == 0 and ")" in stop:
                return i
            depth -= 1
        elif c in stop and depth == 0:
            return i
        i += 1
    return n


_WORD = re.compile(r"[A-Za-z_$][A-Za-z0-9_$]*")
_TYPE_WORDS = re.compile(r"[A-Za-z_][A-Za-z0-9_]*(?:\s+(?:PRECISION|VARYING)\b)?",
                         re.IGNORECASE)
_TABLE_CONSTRAINT_WORDS = {"CONSTRAINT", "PRIMARY", "UNIQUE", "FOREIGN"}
# A clustering clause that precedes the column list:
# `create or replace TABLE T cluster by (K)(cols...)`, or `LINEAR(K)`.
_CLUSTER_BY_TAIL = re.compile(r"\bcluster\s+by(?:\s+linear)?\s*$",
                              re.IGNORECASE)


def _column_list_opener(text: str) -> int:
    """Index of the "(" that opens the column list, or len(text).

    The first top-level "(" is the column list unless it belongs to a
    `cluster by (...)` written before it -- the order sqlglot's Snowflake
    dialect accepts, and likely GET_DDL's for a clustered table (not
    captured live). That group, nested expression keys included, is
    skipped; the clause written after the list needs nothing.
    """
    pos = 0
    while True:
        opener = _scan_to(text, pos, "(")
        if opener >= len(text):
            return len(text)
        if not _CLUSTER_BY_TAIL.search(text[pos:opener]):
            return opener
        pos = _scan_to(text, opener + 1, ")") + 1


def _ddl_column_types(ddl: str) -> dict[str, str]:
    """{column: declared type} from GET_DDL('TABLE') text.

    Only the type is taken -- its first word(s) and one balanced
    parenthesised group, `MAP(VARCHAR(16777216), NUMBER(38,0))` -- never the
    NOT NULL / DEFAULT / COMMENT after it, whose literals may hold commas
    and parentheses. A table constraint line (`primary key (ID)`) is not a
    column. A `cluster by (...)` before the column list is skipped. Text it
    cannot read yields what it could, possibly nothing.
    """
    text = ddl or ""
    opener = _column_list_opener(text)
    if opener >= len(text):
        return {}
    out: dict[str, str] = {}
    pos = opener + 1
    while pos < len(text):
        end = _scan_to(text, pos, ",)")
        item = text[pos:end].strip()
        pos = end + 1
        name, rest = None, ""
        if item.startswith('"'):
            close = 1
            while close < len(item):
                if item[close] == '"':
                    if item[close:close + 2] == '""':
                        close += 2
                        continue
                    break
                close += 1
            name, rest = item[1:close].replace('""', '"'), item[close + 1:]
        elif item:
            m = _WORD.match(item)
            if m and m.group(0).upper() not in _TABLE_CONSTRAINT_WORDS:
                name, rest = m.group(0), item[m.end():]
        rest = rest.lstrip()
        m = _TYPE_WORDS.match(rest) if name is not None else None
        if m:
            type_end = m.end()
            after = rest[type_end:].lstrip()
            if after.startswith("("):
                offset = len(rest) - len(after)
                type_end = min(_scan_to(rest, offset + 1, ")") + 1, len(rest))
            out[name] = rest[:type_end].strip()
        if end >= len(text) or text[end] == ")":
            break
    return out


def _qualified_literal(database: str, schema: str, table: str) -> str:
    """'"DB"."SCHEMA"."TABLE"' as a Snowflake string literal, for GET_DDL.

    The database is the config's value resolved the way every other
    pushdown resolves it (`_database_name`): quoted verbatim, a config's
    `snowmig_db` named a lower-case database that does not exist, and every
    type-detail read failed while the counts and copies worked."""
    name = ".".join('"' + str(p).replace('"', '""') + '"'
                    for p in (_database_name(database), schema, table))
    return "'" + _sql_literal(name) + "'"


def _get_ddl_sql(database: str, batch: list[tuple[str, str]]) -> str:
    return " union all ".join(
        f"select '{_sql_literal(schema)}' as SNOWMIG_SCHEMA, "
        f"'{_sql_literal(table)}' as SNOWMIG_TABLE, "
        f"get_ddl('table', {_qualified_literal(database, schema, table)}) "
        f"as SNOWMIG_DDL" for schema, table in batch)


def read_type_details(source: SnowflakeSource,
                      by_object: dict[tuple[str, str], list[dict]],
                      tables: list[tuple[str, str]]) -> None:
    """Record `type_detail` on every column that needs it, or say why not.

    There is no DESCRIBE through the connector: its pushdown takes a query.
    GET_DDL is a SELECT, its CREATE TABLE text spells the same full types
    DESCRIBE does, and many calls ride one UNION ALL -- fully qualified,
    because the pushdown session has no current schema and an unqualified
    name fails live ("Object does not exist"). A chunk that fails is retried
    table by table, so one unreadable table does not cost the other 49.
    Where nothing can be read the column carries `type_detail_unread`; the
    engine's mapper then treats the element types as unknown and says so.
    """
    def needing(key):
        return [c for c in by_object.get(key, [])
                if _needs_type_detail(c.get("data_type"),
                                      c.get("datetime_precision"))]

    wanted = [key for key in tables if needing(key)]
    if not wanted:
        return

    def mark(key, reason):
        for col in needing(key):
            col.setdefault("type_detail_unread", reason)

    try:
        database = source.database()
    except Exception:
        database = None
    if not database:
        for key in wanted:
            mark(key, "no source database is configured to qualify the "
                      "GET_DDL read with, and an unqualified name fails "
                      "through the pushdown; element types UNREAD")
        log(f"type detail: {len(wanted)} table(s) need it and no database "
            f"is configured to qualify GET_DDL; recorded as UNREAD")
        return

    def run(batch):
        rows = _rows(source.pushdown(_get_ddl_sql(database, batch)))
        return {(str(r.get("SNOWMIG_SCHEMA")), str(r.get("SNOWMIG_TABLE"))):
                r.get("SNOWMIG_DDL") for r in rows}

    ddls: dict[tuple[str, str], object] = {}
    for start in range(0, len(wanted), _DDL_CHUNK):
        batch = wanted[start:start + _DDL_CHUNK]
        try:
            ddls.update(run(batch))
            continue
        except Exception as exc:
            if len(batch) == 1:
                mark(batch[0], f"GET_DDL failed: {str(exc)[:200]}")
                continue
        for key in batch:
            try:
                ddls.update(run([key]))
            except Exception as exc:
                mark(key, f"GET_DDL failed: {str(exc)[:200]}")

    read = 0
    for key in wanted:
        if key not in ddls:
            mark(key, "GET_DDL returned no row for this table")
            continue
        types = _ddl_column_types(str(ddls[key] or ""))
        for col in needing(key):
            detail = types.get(col["name"])
            if detail:
                col["type_detail"] = detail
                col.pop("type_detail_unread", None)
                read += 1
            else:
                col.setdefault("type_detail_unread",
                               "the GET_DDL text carried no type for this "
                               "column that could be read")
    log(f"type detail: {read} column(s) read through GET_DDL for "
        f"{len(wanted)} table(s)")


def discover_via_connector(source: SnowflakeSource, *,
                           wanted: list[str] | None,
                           exclude: set[str]) -> list[dict]:
    """Every schema/table/column of the database (or of `wanted`), in two
    queries."""
    # INFORMATION_SCHEMA is per-database. The connector's `schema` option
    # must name a REAL schema (it rejects INFORMATION_SCHEMA itself with
    # DATA_ACCESS_LAYER_0031); the SQL below then reads INFORMATION_SCHEMA
    # relative to the database. Both established live.
    where = _schema_predicate(wanted)
    flags_unread = None
    try:
        tables = _rows(source.pushdown(
            _TABLES_SQL.format(where=where, flags=_KIND_FLAGS_SELECT)))
    except Exception as exc:
        if "invalid identifier" not in str(exc).lower():
            raise
        flags_unread = str(exc)[:300]
        log(f"INFORMATION_SCHEMA.TABLES has no {'/'.join(_KIND_FLAGS)} here "
            f"({flags_unread}); re-reading without them. Dynamic, Iceberg "
            f"and hybrid tables are NOT told apart from base tables in "
            f"this manifest")
        tables = _rows(source.pushdown(
            _TABLES_SQL.format(where=where, flags="")))
    columns = _rows(source.pushdown(_COLUMNS_SQL.format(where=where)))
    log(f"INFORMATION_SCHEMA: {len(tables)} relation(s), "
        f"{len(columns)} column(s), in {3 if flags_unread else 2} queries"
        + (f" scoped to {len(wanted)} schema(s)" if wanted else ""))

    by_object: dict[tuple[str, str], list[dict]] = {}
    for col in columns:
        key = (str(col["TABLE_SCHEMA"]), str(col["TABLE_NAME"]))
        by_object.setdefault(key, []).append(
            # The formatted `type` is for humans reading the manifest. The
            # four raw INFORMATION_SCHEMA fields below are what the migrator's
            # type mapper consumes, and they are carried verbatim: re-parsing
            # "NUMBER(38,0)" back into precision and scale would be a second,
            # lossier implementation of something we already have exactly.
            {"name": str(col["COLUMN_NAME"]),
             "type": _snowflake_type(col),
             "nullable": str(col.get("IS_NULLABLE") or "").upper() != "NO",
             "ordinal_position": _plain(col.get("ORDINAL_POSITION")),
             "data_type": _plain(col.get("DATA_TYPE")),
             "numeric_precision": _plain(col.get("NUMERIC_PRECISION")),
             "numeric_scale": _plain(col.get("NUMERIC_SCALE")),
             "character_maximum_length": _plain(
                 col.get("CHARACTER_MAXIMUM_LENGTH")),
             # A collated text column compares differently once it is a
             # Delta STRING; the engine's mapper warns on it.
             "collation": _plain(col.get("COLLATION_NAME")),
             # Precision 9 (Snowflake's default) loses three digits at the
             # Spark read; the mapper warns on anything above 6.
             "datetime_precision": _plain(col.get("DATETIME_PRECISION")),
             # What R22/R23 and the column comment read. Selecting them was
             # half a fix: until they were written here too, every manifest
             # said "facts unknown" and a re-run could never change that.
             "column_default": _plain(col.get("COLUMN_DEFAULT")),
             "identity_start": _plain(col.get("IDENTITY_START")),
             "identity_increment": _plain(col.get("IDENTITY_INCREMENT")),
             "comment": _plain(col.get("COMMENT")),
             # Read, so a None above is a real "none", never "unknown".
             "facts_recorded": True})

    # Views are not copied, so only a base table's structured columns are
    # worth the extra read.
    read_type_details(source, by_object, [
        (str(rel["TABLE_SCHEMA"]), str(rel["TABLE_NAME"])) for rel in tables
        if "VIEW" not in str(rel.get("TABLE_TYPE") or "BASE TABLE").upper()
        and str(rel["TABLE_SCHEMA"]).lower() not in exclude
        and not (wanted and str(rel["TABLE_SCHEMA"]) not in wanted)])

    schemas: dict[str, dict] = {}
    for rel in tables:
        schema = str(rel["TABLE_SCHEMA"])
        if schema.lower() in exclude or (wanted and schema not in wanted):
            continue
        name = str(rel["TABLE_NAME"])
        kind = str(rel.get("TABLE_TYPE") or "BASE TABLE").upper()
        record = schemas.setdefault(
            schema, {"name": schema, "tables": [], "views": [], "errors": []})
        entry = {"name": name,
                 "columns": by_object.get((schema, name), []),
                 "source_rows": _plain(rel.get("ROW_COUNT")),
                 "source_bytes": _plain(rel.get("BYTES")),
                 # Recorded as Snowflake reports them; the bridge maps them
                 # onto the planner's kind flags.
                 "table_type": kind}
        for field in ("IS_TRANSIENT", "COMMENT") + _KIND_FLAGS:
            if rel.get(field) is not None:
                entry[field.lower()] = _plain(rel.get(field))
        if flags_unread:
            entry["kind_flags_unread"] = flags_unread
        if not entry["columns"]:
            record["errors"].append(
                {"object": name, "kind": kind,
                 "error": "INFORMATION_SCHEMA.COLUMNS returned no column for "
                          "this relation, so it is INCOMPLETE, not empty"})
        bucket = record["views"] if "VIEW" in kind else record["tables"]
        bucket.append(entry)
    return [schemas[k] for k in sorted(schemas)]


def _describe(source: SnowflakeSource, schema: str, name: str) -> list[dict]:
    """Ordered [{name, type}] for one object. DESCRIBE emits section markers
    (`# Partitioning`, blank names) after the column list; stop at the first."""
    columns: list[dict] = []
    fqn = f"{q(source.external_catalog)}.{q(schema)}.{q(name)}"
    for row in _rows(source.spark.sql(f"DESCRIBE {fqn}")):
        col = str(row.get("col_name") or "").strip()
        if not col or col.startswith("#"):
            break
        columns.append({"name": col, "type": str(row.get("data_type") or "")})
    return columns


def discover_schema_via_catalog(source: SnowflakeSource, schema: str) -> dict:
    out = {"name": schema, "tables": [], "views": [], "errors": []}
    catalog = q(source.external_catalog)

    for row in _rows(source.spark.sql(f"SHOW TABLES IN {catalog}.{q(schema)}")):
        name = str(_first_key(row, "tableName", "table_name", "name"))
        try:
            out["tables"].append(
                {"name": name, "columns": _describe(source, schema, name)})
        except Exception as exc:  # one object must not eat the schema
            out["errors"].append({"object": name, "kind": "TABLE",
                                  "error": str(exc)[:300]})

    # SHOW VIEWS may not be supported against every external catalog; an
    # unreadable view list is RECORDED, never silently read as "no views".
    try:
        for row in _rows(source.spark.sql(
                f"SHOW VIEWS IN {catalog}.{q(schema)}")):
            name = str(_first_key(row, "viewName", "view_name", "name"))
            try:
                columns = _describe(source, schema, name)
            except Exception as exc:
                columns = []
                out["errors"].append({"object": name, "kind": "VIEW",
                                      "error": str(exc)[:300]})
            out["views"].append({"name": name, "columns": columns})
    except Exception as exc:
        out["errors"].append({"object": "*", "kind": "VIEW_LIST",
                              "error": f"SHOW VIEWS failed: {str(exc)[:300]}"})
    return out


def _unanswered_schemas(existing: list[dict], requested: set[str],
                        fresh: list[dict], exclude: set[str], *,
                        force: bool, keep: list[dict]) -> int:
    """Failures for --schemas names that returned no rows; the kept entries
    are appended to `keep`.

    Zero rows is not "the schema is gone". After a grant revocation
    INFORMATION_SCHEMA returns nothing, with no error, and a misspelt or
    wrongly-cased name does the same. The scoped merge used to drop the
    named schema and add nothing, so SALES vanished from the manifest --
    and from reconcile, failed copy and all -- with exit 0. Now the prior
    discovery is kept with the reason attached and the run fails; --force
    is the deliberate drop.
    """
    returned = {s["name"] for s in fresh}
    prior = {s["name"]: s for s in existing}
    failures = 0
    for name in sorted(requested - returned):
        if name.lower() in exclude:
            continue
        alike = sorted(n for n in prior if n.lower() == name.lower()
                       and n != name)
        hint = (f"; schema names are case-sensitive, and the manifest has "
                f"{', '.join(alike)}" if alike else "")
        if force:
            log(f"--schemas {name} returned no rows and --force was given: "
                + ("dropped from the manifest" if name in prior
                   else "nothing of that name to drop") + hint)
            continue
        failures += 1
        reason = (f"re-discovery with --schemas {name} returned no rows: not "
                  f"visible to this role, empty, or misspelled{hint}. The "
                  f"previous discovery is kept; --force drops it")
        log(f"SCHEMA RETURNED NO ROWS: {reason}")
        if name in prior:
            entry = dict(prior[name])
            entry["errors"] = [e for e in entry.get("errors") or []
                               if e.get("kind") != "REDISCOVERY"] + [
                {"object": "*", "kind": "REDISCOVERY", "error": reason}]
            keep.append(entry)
    return failures


def render_summary(manifest: dict) -> str:
    src = manifest.get("source") or {}
    where = (f'catalog `{src.get("external_catalog")}`'
             if src.get("mode") == "external-catalog"
             else f'`{src.get("host")}` / `{src.get("database")}` as '
                  f'`{src.get("user")}`')
    lines = ["# Discovery — what the Snowflake source exposes", "",
             f'Mode: `{src.get("mode")}` · {where}',
             f'{len(manifest["schemas"])} schema(s) · generated '
             f'{manifest.get("generated_at", "?")}', "",
             "| Schema | Tables | Views | Columns | Errors |",
             "|---|---|---|---|---|"]
    for s in manifest["schemas"]:
        cols = sum(len(t["columns"]) for t in s["tables"] + s["views"])
        mark = " ⚠️" if s["errors"] else ""
        lines.append(f'| {s["name"]} | {len(s["tables"])} | {len(s["views"])} '
                     f'| {cols} | {len(s["errors"])}{mark} |')
    lines += ["",
              "A schema with errors is INCOMPLETE, not empty — re-run with "
              "`--schemas <name>` (add `--force` in external-catalog mode) after "
              "fixing the cause; the other schemas are kept.", ""]
    return "\n".join(lines)


def backup_manifest(manifest: dict, backup_dir: str,
                    now: datetime.datetime | None = None) -> pathlib.Path | None:
    """Write the dated copy of the manifest into `backup_dir`; '' skips.

    Named by the UTC time of the run, never overwritten: every discovery
    leaves its own copy, so the input to a plan can be found again after a
    later re-run has merged new schemas into reports/.
    """
    if not backup_dir:
        return None
    stamp = (now or datetime.datetime.now(datetime.timezone.utc)
             ).strftime("%Y%m%dT%H%M%SZ")
    folder = pathlib.Path(backup_dir)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{pathlib.Path(MANIFEST_NAME).stem}_{stamp}.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return path


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--source-mode", choices=list(SOURCE_MODES),
                    default="connector",
                    help="connector (default): read Snowflake directly from "
                         "the cluster — one INFORMATION_SCHEMA query pair for "
                         "the whole estate. external-catalog: SHOW/DESCRIBE "
                         "against a registered EXTERNAL catalog, which needs "
                         "a successful crawl")
    ap.add_argument("--source-config",
                    help="JSON/YAML connection config (connector mode)")
    ap.add_argument("--source-catalog",
                    help="the registered EXTERNAL catalog "
                         "(external-catalog mode)")
    ap.add_argument("--session-schema",
                    help="a REAL schema used only to scope the connector's "
                         "pushdown session (default: `schema` from the source "
                         "config). The connector refuses INFORMATION_SCHEMA "
                         "here, so one real schema name is required")
    ap.add_argument("--schemas", nargs="*", default=None,
                    help="only these schemas (default: all). Pushed into the "
                         "INFORMATION_SCHEMA queries as a predicate, and "
                         "merged into the existing manifest: the other "
                         "schemas are kept")
    ap.add_argument("--exclude-schemas", nargs="*", default=[])
    ap.add_argument("--output-dir", default=DEFAULT_OUTPUT_DIR,
                    help="where this step saves its values (report/output in "
                         "the workspace); '' to skip")
    ap.add_argument("--reports-dir", default=DEFAULT_REPORTS_DIR)
    ap.add_argument("--backup-dir", default=None,
                    help="where the dated copy of the manifest is written "
                         "(runbook S6). Default: the backup/ folder beside "
                         "--reports-dir; '' to skip")
    ap.add_argument("--force", action="store_true",
                    help="rediscover the named --schemas even if the manifest "
                         "already carries them (the others are kept), and "
                         "drop a named schema that returns no rows instead "
                         "of keeping it and failing; without --schemas, "
                         "rediscover the whole estate")
    args = ap.parse_args(argv)

    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    try:
        config = (load_source_config(args.source_config)
                  if args.source_config else None)
        source = SnowflakeSource(spark, mode=args.source_mode, config=config,
                                 external_catalog=args.source_catalog,
                                 session_schema=args.session_schema)
    except SourceConfigError as exc:
        return fail(f"error: {exc}")

    reports = pathlib.Path(args.reports_dir)
    reports.mkdir(parents=True, exist_ok=True)
    manifest_path = reports / MANIFEST_NAME
    identity = (source.external_catalog if args.source_mode
                == "external-catalog" else source.database())

    manifest = {"schemas": []}
    if manifest_path.exists():
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        # The identity guard holds under --force too: a --force against a
        # manifest written for another database must not overwrite it.
        if existing.get("source_identity") not in (None, identity):
            return fail(f"error: {manifest_path} describes "
                  f"{existing.get('source_identity')!r}, not {identity!r}. "
                  f"Use a fresh --reports-dir.")
        if args.force and not args.schemas:
            log("--force without --schemas: rediscovering the whole estate")
        else:
            manifest = existing

    manifest.update(source=source.describe(), source_identity=identity,
                    generated_at=datetime.datetime.now(
                        datetime.timezone.utc).isoformat())
    log(f"source: {json.dumps(source.describe())}")

    exclude = {s.lower() for s in args.exclude_schemas} | _SYSTEM_SCHEMAS
    failures = 0

    if args.source_mode == "connector":
        try:
            fresh = discover_via_connector(
                source, wanted=args.schemas, exclude=exclude)
        except Exception:
            failures += 1
            log(f"DISCOVERY FAILED:\n{traceback.format_exc(limit=3)}")
        else:
            if args.schemas:
                # A scoped run MERGES. Assigning the filtered result over the
                # loaded manifest used to drop every other schema -- exactly
                # what DISCOVERY.md's own "re-run with --schemas <name>"
                # advice then did to a finished discovery. An unscoped run is
                # the whole estate and stays authoritative.
                requested = set(args.schemas)
                kept = [s for s in manifest["schemas"]
                        if s["name"] not in requested]
                failures += _unanswered_schemas(
                    manifest["schemas"], requested, fresh, exclude,
                    force=args.force, keep=kept)
                fresh = sorted(kept + fresh, key=lambda s: s["name"])
            manifest["schemas"] = fresh
    else:
        done = {s["name"] for s in manifest["schemas"]}
        listed = [str(_first_key(r, "namespace", "databaseName",
                                 "schema_name", "name"))
                  for r in _rows(spark.sql(
                      f"SHOW SCHEMAS IN {q(source.external_catalog)}"))]
        wanted = [s for s in (args.schemas or listed)
                  if s.lower() not in exclude]
        log(f"{len(wanted)} schema(s) to discover; {len(done)} already done")
        for schema in wanted:
            if schema in done and not args.force:
                log(f"skip {schema}: already discovered (--force to redo)")
                continue
            try:
                record = discover_schema_via_catalog(source, schema)
            except Exception:
                failures += 1
                log(f"SCHEMA FAILED {schema}:\n{traceback.format_exc(limit=3)}")
                record = {"name": schema, "tables": [], "views": [],
                          "errors": [{"object": "*", "kind": "SCHEMA",
                                      "error": traceback.format_exc(
                                          limit=1)[-300:]}]}
            manifest["schemas"] = sorted(
                [s for s in manifest["schemas"] if s["name"] != schema]
                + [record], key=lambda s: s["name"])
            # Flush after EVERY schema: a large estate resumes, not restarts.
            manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            log(f"{schema}: {len(record['tables'])} table(s), "
                f"{len(record['views'])} view(s)")

    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (reports / "DISCOVERY.md").write_text(render_summary(manifest), encoding="utf-8")
    total = sum(len(s["tables"]) for s in manifest["schemas"])
    log(f"{len(manifest['schemas'])} schema(s), {total} table(s) "
        f"-> {manifest_path}")
    log(f"summary -> {reports / 'DISCOVERY.md'}")
    backup_dir = (str(reports.parent / BACKUP_FOLDER_NAME)
                  if args.backup_dir is None else args.backup_dir)
    backup_path = backup_manifest(manifest, backup_dir)
    if backup_path:
        log(f"backup -> {backup_path}")
    write_step_output(args.output_dir, "S06_discover.json", {
        "step": "S06", "stage": "discover",
        "schemas": len(manifest["schemas"]), "tables": total,
        "views": sum(len(s["views"]) for s in manifest["schemas"]),
        "failures": failures, "manifest": str(manifest_path),
        "backup": str(backup_path) if backup_path else None})

    if not manifest["schemas"]:
        log("ZERO schemas discovered. In external-catalog mode that usually "
            "means the catalog has not completed a crawl yet (check its "
            "refresh status and the crawler's network path to Snowflake); in "
            "connector mode it means these credentials see nothing, or "
            "DISCOVERY FAILED above (read that traceback first). Either "
            "way it is a FINDING, not a success.")
        return 1
    return 1 if failures else 0



In [ ]:
# ── RUN ────────────────────────────────────────────────────────────
# main() RETURNS an exit code; it is not allowed to raise SystemExit here.
# A notebook cell that raises SystemExit is reported as a FAILED task even
# when the work succeeded, so the code is inspected and only a real failure
# is re-raised -- which keeps a genuinely failed stage failing.
code = main(ARGV)
print('exit code:', code, flush=True)
if code:
    raise RuntimeError(f'stage exited {code}')
